In [7]:
import pandas as pd
import duckdb

df = pd.read_parquet("../data/processed/steam_clean.parquet")

df[["price", "total_reviews", "pct_positive"]].describe().round(2)

,price,total_reviews,pct_positive
count,55942.00,55942.00,55942.00
mean,5.13,2647.67,0.77
std,10.28,48273.23,0.18
min,0.00,10.00,0.00
25%,0.99,22.00,0.67
50%,2.99,60.00,0.81
75%,6.24,284.00,0.91
max,199.99,8815087.00,1.00


In [2]:
query = """
SELECT
    CASE
        WHEN price = 0  THEN '1. Free'
        WHEN price < 5  THEN '2. Under $5'
        WHEN price < 10 THEN '3. $5-9.99'
        WHEN price < 20 THEN '4. $10-19.99'
        WHEN price < 40 THEN '5. $20-39.99'
        ELSE '6. $40+'
    END AS price_band,
    COUNT(*)                              AS games,
    ROUND(MEDIAN(pct_positive) * 100, 1)  AS median_pct_positive,
    MEDIAN(total_reviews)                 AS median_reviews
FROM df
GROUP BY price_band
ORDER BY price_band
"""
duckdb.sql(query).df()

,price_band,games,median_pct_positive,median_reviews
0,1. Free,7368,78.3,62.0
1,2. Under $5,32605,81.0,48.0
2,3. $5-9.99,9958,83.7,78.0
3,4. $10-19.99,5403,83.0,131.0
4,5. $20-39.99,950,81.2,359.0
5,6. $40+,275,69.0,45.0


In [3]:
known = ["Stardew Valley", "Terraria", "Hollow Knight", "Hades",
         "ELDEN RING", "Baldur's Gate 3", "Celeste", "Cyberpunk 2077"]
df[df["name"].isin(known)][["name", "price", "release_date"]]

,name,price,release_date
5064,Terraria,4.99,2011-05-16
22576,Celeste,4.99,2018-01-25
27548,Baldur's Gate 3,44.99,2023-08-03
31971,Cyberpunk 2077,20.99,2020-12-09
32599,Hades,7.49,2020-09-17
35808,Hollow Knight,7.49,2017-02-24
44440,Stardew Valley,8.99,2016-02-26
53153,ELDEN RING,38.99,2022-02-24


In [4]:
import requests

r = requests.get("https://steamspy.com/api.php",
                 params={"request": "all", "page": 0}, timeout=60)
page0 = r.json()
print("Games on this page:", len(page0))

# check the known games (Steam appIDs)
known_ids = [105600, 504230, 1145360, 367520, 413150, 1091500, 1245620, 1086940]
for appid in known_ids:
    g = page0.get(str(appid))
    if g:
        print(g["name"], "| price:", g.get("price"),
              "| initialprice:", g.get("initialprice"), "| discount:", g.get("discount"))
    else:
        print(appid, "not on page 0")

Games on this page: 1000
Terraria | price: 999 | initialprice: 999 | discount: 0
Celeste | price: 499 | initialprice: 1999 | discount: 75
1145360 not on page 0
Hollow Knight | price: 1499 | initialprice: 1499 | discount: 0
Stardew Valley | price: 1499 | initialprice: 1499 | discount: 0
1091500 not on page 0
1245620 not on page 0
1086940 not on page 0


**Data issue found:** the `price` column reflects sale prices on the day the data
was collected, not normal prices (e.g. Celeste $4.99 vs. $19.99 normal, Hades $7.49
vs. $24.99). The price-band results above are therefore invalid. Fix: replace with
each game's pre-discount price from SteamSpy.

In [8]:
genres_long = df[["appID", "name", "genres", "pct_positive", "total_reviews"]].explode("genres")
genres_long = genres_long.rename(columns={"genres": "genre"})

print("Games:", df.shape[0], "→ game-genre rows:", genres_long.shape[0])

Games: 55942 → game-genre rows: 161651


In [9]:
query = """
SELECT
    genre,
    COUNT(*)                              AS games,
    ROUND(MEDIAN(pct_positive) * 100, 1)  AS median_pct_positive,
    MEDIAN(total_reviews)                 AS median_reviews
FROM genres_long
GROUP BY genre
HAVING COUNT(*) >= 100
ORDER BY median_pct_positive DESC
"""
duckdb.sql(query).df()

,genre,games,median_pct_positive,median_reviews
0,Casual,21802,82.4,44.0
1,Indie,40580,81.8,54.0
2,Adventure,23959,81.3,63.0
3,Design & Illustration,114,81.1,49.0
4,RPG,10950,80.0,94.0
5,Action,23894,80.0,60.0
6,Strategy,11612,79.0,71.0
7,Free To Play,4230,78.3,97.0
8,Racing,2053,78.1,54.0
9,Sports,2540,77.6,53.0


In [10]:
gameplay = ["Action", "Adventure", "Casual", "RPG", "Strategy",
            "Simulation", "Racing", "Sports", "Massively Multiplayer"]
genre_list = ", ".join(f"'{g}'" for g in gameplay)

query = f"""
SELECT
    genre,
    COUNT(*)                                          AS games,
    ROUND(QUANTILE_CONT(pct_positive, 0.25) * 100, 1) AS p25,
    ROUND(MEDIAN(pct_positive) * 100, 1)              AS median,
    ROUND(QUANTILE_CONT(pct_positive, 0.75) * 100, 1) AS p75,
    MEDIAN(total_reviews)                             AS median_reviews
FROM genres_long
WHERE genre IN ({genre_list})
GROUP BY genre
ORDER BY median DESC
"""
duckdb.sql(query).df()

,genre,games,p25,median,p75,median_reviews
0,Casual,21802,68.0,82.4,92.1,44.0
1,Adventure,23959,67.5,81.3,90.9,63.0
2,RPG,10950,67.1,80.0,90.0,94.0
3,Action,23894,66.4,80.0,90.3,60.0
4,Strategy,11612,65.3,79.0,89.1,71.0
5,Racing,2053,63.2,78.1,89.0,54.0
6,Sports,2540,63.6,77.6,88.0,53.0
7,Simulation,12657,62.4,77.3,88.3,74.0
8,Massively Multiplayer,1403,54.5,66.7,77.8,141.0


**Finding (genre):** Among gameplay genres, reception is broadly similar (median 77–82% positive, with heavily overlapping middle-50% ranges). Massively Multiplayer is the exception (median 66.7%, upper quartile below most genres' medians). Genre is more strongly associated with reach than reception: RPG and MMO games have 2–3x the median review count of Casual games. These are associations; MMO results may reflect live-service factors (long review windows, updates, monetization) rather than the genre itself.

In [11]:
cats_long = df[["appID", "categories", "pct_positive", "total_reviews"]].explode("categories")
cats_long = cats_long.rename(columns={"categories": "category"})

query = """
SELECT
    category,
    COUNT(*)                                          AS games,
    ROUND(QUANTILE_CONT(pct_positive, 0.25) * 100, 1) AS p25,
    ROUND(MEDIAN(pct_positive) * 100, 1)              AS median,
    ROUND(QUANTILE_CONT(pct_positive, 0.75) * 100, 1) AS p75,
    MEDIAN(total_reviews)                             AS median_reviews
FROM cats_long
GROUP BY category
HAVING COUNT(*) >= 500
ORDER BY games DESC
"""
duckdb.sql(query).df()

,category,games,p25,median,p75,median_reviews
0,Single-player,53590,67.9,81.8,91.3,59.0
1,Family Sharing,49021,67.6,81.8,91.5,59.0
2,Steam Achievements,33086,71.2,83.3,92.1,84.0
3,Steam Cloud,19413,74.4,85.3,92.9,142.0
4,Full controller support,15499,74.5,85.6,93.3,92.0
5,Multi-player,11785,65.3,78.6,88.8,120.0
6,Steam Trading Cards,10505,68.4,80.3,89.2,375.0
7,Partial Controller Support,7913,67.4,81.0,90.8,79.0
8,PvP,6822,65.6,78.6,88.8,91.0
9,Co-op,6582,66.7,79.7,89.4,131.5


In [13]:
online_cats = {"Online PvP", "Online Co-op", "MMO", "Cross-Platform Multiplayer"}
local_cats = {"Shared/Split Screen", "Shared/Split Screen PvP", "Shared/Split Screen Co-op",
              "LAN Co-op", "LAN PvP", "Remote Play Together"}
general_mp = {"Multi-player", "PvP", "Co-op"}

def play_mode(categories):
    cats = set(categories)
    if cats & online_cats:
        return "1. Online multiplayer"
    if cats & local_cats:
        return "2. Local multiplayer only"
    if cats & general_mp:
        return "3. Multiplayer (type not specified)"
    if "Single-player" in cats:
        return "4. Single-player only"
    return "5. Unclear"

df["play_mode"] = df["categories"].apply(play_mode)

query = """
SELECT
    play_mode,
    COUNT(*)                                          AS games,
    ROUND(QUANTILE_CONT(pct_positive, 0.25) * 100, 1) AS p25,
    ROUND(MEDIAN(pct_positive) * 100, 1)              AS median,
    ROUND(QUANTILE_CONT(pct_positive, 0.75) * 100, 1) AS p75,
    MEDIAN(total_reviews)                             AS median_reviews
FROM df
GROUP BY play_mode
ORDER BY play_mode
"""
duckdb.sql(query).df()

,play_mode,games,p25,median,p75,median_reviews
0,1. Online multiplayer,7383,64.0,76.8,87.3,157.0
1,2. Local multiplayer only,3704,71.3,83.3,92.0,54.0
2,3. Multiplayer (type not specified),1124,59.1,73.6,85.8,380.5
3,4. Single-player only,43443,68.3,82.1,91.7,52.0
4,5. Unclear,288,60.0,77.2,88.5,42.0


**Finding (play mode):** Games with online multiplayer have lower reception (median 76.8% positive) than single-player-only (82.1%) or local-multiplayer-only games (83.3%), and about 3x the median review count (157 vs 52). Local multiplayer shows no such gap, so the association is with online play specifically. The difference is moderate: middle-50% ranges overlap substantially. Likely confounders include live-service factors such as server stability, monetization and long-running review windows.

In [14]:
import requests

r = requests.get("https://store.steampowered.com/api/appdetails",
                 params={"appids": 504230, "cc": "us", "filters": "price_overview"},
                 timeout=30)
print(r.json())

{'504230': {'success': True, 'data': {'price_overview': {'currency': 'USD', 'initial': 1999, 'final': 499, 'discount_percent': 75, 'initial_formatted': '$19.99', 'final_formatted': '$4.99'}}}}
